# 07 — Pre-registered Correspondential Tests

Implements `docs/PREREGISTRATION_2026-10.md` exactly as registered. The registration was committed to git (commit `b3f91962`) before this notebook was written or run.

| # | Prediction | Family |
|---|---|---|
| P1 | Locations with noxious animals are more often negative than locations with gentle animals | Primary (Holm) |
| P2 | Locations with noxious animals are more often underground than those with gentle animals | Secondary |
| P3 | Locations with a deceased person are less often negative than locations with living known persons only | Primary (Holm) |
| P4 | Authors meet consistent atmospheres across different dreams (ICC > 0) | Primary (Holm) |

All tests are one-sided in the predicted direction at α = 0.05, with Holm correction across P1, P3 and P4. Sensitivity analyses and any other comparison are labelled as such and do not change a verdict.

In [1]:
import sys, json, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency, spearmanr, fisher_exact
from statsmodels.stats.multitest import multipletests

MW_ROOT = Path.cwd().parent
sys.path.insert(0, str(MW_ROOT / "scripts"))
import mallworld_dataset as mw  # verified loader: schema-checked fields, populations, cluster keys

OUTPUT_DIR = MW_ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

posts = mw.load_posts()
primary = posts[posts["primary"]].copy()
print(f"Posts: {len(posts):,} | primary dream reports: {len(primary):,} from {primary['author'].nunique():,} authors")

import re
L = mw.load_locations()
E = mw.load_entities()
L["neg"] = (L["valence"] == -1).astype(float).where(L["valence"].notna())

GENTLE = """cat cats kitten kittens kitty dog dogs puppy puppies sheep lamb lambs horse horses pony cow cows cattle calf deer fawn
rabbit rabbits bunny bunnies bird birds dove doves chicken chickens duck ducks swan swans fish goldfish dolphin dolphins butterfly
butterflies squirrel squirrels hamster turtle turtles owl owls""".split()
NOXIOUS = """snake snakes serpent serpents spider spiders rat rats mouse mice cockroach cockroaches roach roaches insect insects bug bugs
ant ants wasp wasps hornet hornets scorpion scorpions worm worms maggot maggots leech leeches wolf wolves shark sharks crocodile
crocodiles alligator alligators bear bears lion lions tiger tigers hound hounds hyena hyenas vulture vultures predator predators""".split()
re_g = re.compile(r"\b(" + "|".join(GENTLE) + r")\b")
re_n = re.compile(r"\b(" + "|".join(NOXIOUS) + r")\b")
clean = lambda s: re.sub(r"\bhot ?dogs?\b", " ", (s or "").lower())

# Entity text per (dream, location)
E["text"] = (E["entity_description"].fillna("") + " " + E["name_or_relation"].fillna("")).map(clean)
ent_text = E.groupby(["post_id", "location_id"])["text"].apply(" ".join)
L["ent_text"] = [ent_text.get((p, l), "") for p, l in zip(L["post_id"], L["location_id"])]
L["loc_text"] = L["raw_description"].map(clean)
for src in ["ent", "loc"]:
    L[f"g_{src}"] = L[f"{src}_text"].str.contains(re_g)
    L[f"n_{src}"] = L[f"{src}_text"].str.contains(re_n)
L["gentle"] = L["g_ent"] | L["g_loc"]
L["noxious"] = L["n_ent"] | L["n_loc"]
A = L[L["gentle"] ^ L["noxious"]].copy()          # exactly one class
A["nox"] = A["noxious"].astype(int)
print(f"Locations with gentle animals only: {int((L['gentle'] & ~L['noxious']).sum())}; noxious only: {int((L['noxious'] & ~L['gentle']).sum())}; "
      f"both (excluded): {int((L['gentle'] & L['noxious']).sum())}")
words = Counter()
for t in A["ent_text"] + " " + A["loc_text"]:
    words.update(m.group(1) for m in re.finditer(re_g, t))
    words.update(m.group(1) for m in re.finditer(re_n, t))
print("Most frequent animal words:", words.most_common(25))

Posts: 3,732 | primary dream reports: 1,918 from 1,303 authors


Locations with gentle animals only: 165; noxious only: 55; both (excluded): 6
Most frequent animal words: [('fish', 52), ('dog', 47), ('cat', 37), ('dogs', 27), ('cats', 17), ('horse', 11), ('sharks', 11), ('horses', 10), ('deer', 8), ('birds', 7), ('spider', 7), ('bird', 7), ('chicken', 5), ('owl', 5), ('dolphin', 5), ('dolphins', 5), ('bugs', 4), ('duck', 4), ('bears', 4), ('shark', 4), ('ant', 4), ('chickens', 4), ('rat', 4), ('snake', 4), ('roaches', 3)]


/tmp/ipykernel_3601/1390799915.py:47: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  L[f"g_{src}"] = L[f"{src}_text"].str.contains(re_g)
/tmp/ipykernel_3601/1390799915.py:48: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  L[f"n_{src}"] = L[f"{src}_text"].str.contains(re_n)


## P1 — Noxious vs gentle animals and atmosphere

In [2]:
def one_sided(res, term, direction):
    b, p2 = res.params[term], res.pvalues[term]
    return p2 / 2 if np.sign(b) == direction else 1 - p2 / 2

A1 = A.dropna(subset=["neg"])
print(f"P1 sample: {len(A1)} locations in {A1['post_id'].nunique()} dreams (noxious {int(A1['nox'].sum())}, gentle {int((A1['nox'] == 0).sum())})")
rates = A1.groupby("nox").agg(n=("neg", "size"), dreams=("post_id", "nunique"), neg=("neg", "mean"))
rates.index = rates.index.map({0: "gentle", 1: "noxious"})
print((rates.assign(neg=100 * rates["neg"])).round(1).to_string())
res1 = mw.clustered_logit(A1, "neg ~ nox + log_words")
t1 = mw.odds_ratio_table(res1, ["nox"]).iloc[0]
p1 = one_sided(res1, "nox", +1)
print(f"P1: OR = {t1.OR:.2f} [{t1.lo:.2f}-{t1.hi:.2f}], one-sided p = {p1:.3g}")

P1 sample: 129 locations in 105 dreams (noxious 33, gentle 96)
          n  dreams   neg
nox                      
gentle   96      81  58.3
noxious  33      27  84.8


P1: OR = 4.08 [1.20-13.90], one-sided p = 0.0124


## P2 — Noxious vs gentle animals and height (secondary)

In [3]:
A2 = A.dropna(subset=["vertical_level"]).copy()
A2["under"] = (A2["vertical_level"] < 0).astype(float)
n_nox_v = int(A2["nox"].sum())
print(f"P2 sample: {len(A2)} located animal locations (noxious {n_nox_v}, gentle {len(A2) - n_nox_v})")
print(A2.groupby("nox")["under"].agg(["size", "mean"]).rename(index={0: "gentle", 1: "noxious"}).round(3).to_string())
if n_nox_v < 10:
    p2, t2 = np.nan, None
    print("P2: fewer than 10 noxious locations with a vertical level -> UNDERDETERMINED (pre-registered rule)")
else:
    res2 = mw.clustered_logit(A2, "under ~ nox + log_words")
    t2 = mw.odds_ratio_table(res2, ["nox"]).iloc[0]
    p2 = one_sided(res2, "nox", +1)
    print(f"P2: OR = {t2.OR:.2f} [{t2.lo:.2f}-{t2.hi:.2f}], one-sided p = {p2:.3g}")

P2 sample: 75 located animal locations (noxious 20, gentle 55)
         size   mean
nox                 
gentle     55  0.236
noxious    20  0.250


P2: OR = 1.04 [0.31-3.54], one-sided p = 0.473


## P3 — Deceased vs living known persons and atmosphere

In [4]:
types_at = E.groupby(["post_id", "location_id"])["entity_type"].apply(set)
L["types_here"] = [types_at.get((p, l), set()) for p, l in zip(L["post_id"], L["location_id"])]
L["deceased"] = L["types_here"].map(lambda s: "deceased" in s)
L["living_known"] = L["types_here"].map(lambda s: "deceased" not in s and bool(s & {"known_person", "family_member", "friend"}))
D = L[(L["deceased"] | L["living_known"])].dropna(subset=["neg"]).copy()
D["dec"] = D["deceased"].astype(int)
print(f"P3 sample: {len(D)} locations in {D['post_id'].nunique()} dreams (deceased {int(D['dec'].sum())} in {D.loc[D['dec'] == 1, 'post_id'].nunique()} dreams)")
r3 = D.groupby("dec").agg(n=("neg", "size"), dreams=("post_id", "nunique"), neg=("neg", "mean"))
r3.index = r3.index.map({0: "living known only", 1: "deceased present"})
print((r3.assign(neg=100 * r3["neg"])).round(1).to_string())
res3 = mw.clustered_logit(D, "neg ~ dec + log_words")
t3 = mw.odds_ratio_table(res3, ["dec"]).iloc[0]
p3 = one_sided(res3, "dec", -1)
print(f"P3: OR = {t3.OR:.2f} [{t3.lo:.2f}-{t3.hi:.2f}], one-sided p = {p3:.3g}")

P3 sample: 288 locations in 224 dreams (deceased 22 in 18 dreams)
                     n  dreams   neg
dec                                 
living known only  266     210  68.8
deceased present    22      18  45.5
P3: OR = 0.36 [0.15-0.87], one-sided p = 0.012


## P4 — Author consistency of atmosphere across dreams

In [5]:
dream_score = L.dropna(subset=["valence"]).groupby("post_id").agg(score=("valence", "mean"), author=("author", "first"))
multi = dream_score[dream_score["author"].map(dream_score["author"].value_counts()) >= 2].copy()
print(f"P4 sample: {len(multi)} dreams from {multi['author'].nunique()} authors with >= 2 scored dreams")

def icc1(df, group="author", y="score"):
    g = df.groupby(group)[y]
    k = g.size(); n = len(df); a = len(k)
    grand = df[y].mean()
    msb = (k * (g.mean() - grand) ** 2).sum() / (a - 1)
    msw = ((df[y] - g.transform("mean")) ** 2).sum() / (n - a)
    k0 = (n - (k ** 2).sum() / n) / (a - 1)
    return (msb - msw) / (msb + (k0 - 1) * msw)

icc_obs = icc1(multi)
rng = np.random.default_rng(20261006)
perm = np.array([icc1(multi.assign(author=rng.permutation(multi["author"].values))) for _ in range(5000)])
p4 = (1 + (perm >= icc_obs).sum()) / (1 + len(perm))
print(f"P4: ICC1 = {icc_obs:.3f}; permutation null mean {perm.mean():.3f} (95th pct {np.percentile(perm, 95):.3f}); one-sided p = {p4:.4f}")

P4 sample: 563 dreams from 181 authors with >= 2 scored dreams


P4: ICC1 = 0.094; permutation null mean 0.000 (95th pct 0.072); one-sided p = 0.0176


## Verdicts (Holm across P1, P3, P4)

In [6]:
fam = pd.Series({"P1": p1, "P3": p3, "P4": p4})
holm = pd.Series(multipletests(fam.values, method="holm")[1], index=fam.index)
direction_ok = {"P1": t1.OR > 1, "P3": t3.OR < 1, "P4": icc_obs > 0}
rows = []
for k_ in ["P1", "P3", "P4"]:
    verdict = "HIT" if (direction_ok[k_] and holm[k_] < 0.05) else "MISS"
    rows.append({"prediction": k_, "one-sided p": fam[k_], "Holm p": holm[k_], "direction as predicted": direction_ok[k_], "verdict": verdict})
if t2 is None:
    rows.append({"prediction": "P2 (secondary)", "one-sided p": np.nan, "Holm p": np.nan, "direction as predicted": None, "verdict": "UNDERDETERMINED"})
else:
    rows.append({"prediction": "P2 (secondary)", "one-sided p": p2, "Holm p": np.nan, "direction as predicted": t2.OR > 1,
                 "verdict": "HIT" if (t2.OR > 1 and p2 < 0.05) else "MISS"})
verdicts = pd.DataFrame(rows).set_index("prediction")
with pd.option_context("display.float_format", "{:.4g}".format):
    print(verdicts.to_string())

                one-sided p  Holm p  direction as predicted verdict
prediction                                                         
P1                  0.01236 0.03597                    True     HIT
P3                  0.01199 0.03597                    True     HIT
P4                   0.0176 0.03597                    True     HIT
P2 (secondary)       0.4733     NaN                    True    MISS


## Sensitivity analyses (registered; do not change verdicts)

In [7]:
def or_line(df, formula, term):
    r = mw.clustered_logit(df, formula)
    t = mw.odds_ratio_table(r, [term]).iloc[0]
    return f"OR {t.OR:.2f} [{t.lo:.2f}-{t.hi:.2f}], p(two-sided) = {t.p:.3g}, n = {int(r.nobs)}"

top = L["location_type"].value_counts()
A1 = A1.assign(type_c=A1["location_type"].where(A1["location_type"].map(top) >= 40, "rare_type"))
print("P1 + location type:            ", or_line(A1, "neg ~ nox + log_words + C(type_c)", "nox"))
threat_locs = set(E.loc[E["entity_type"] == "threat", ["post_id", "location_id"]].itertuples(index=False, name=None))
A1_nt = A1[[(p_, l_) not in threat_locs for p_, l_ in zip(A1["post_id"], A1["location_id"])]]
print("P1 excluding locations with a 'threat' entity:", or_line(A1_nt, "neg ~ nox + log_words", "nox"))
Ae = L[L["g_ent"] ^ L["n_ent"]].dropna(subset=["neg"]).assign(nox=lambda d: d["n_ent"].astype(int))
print("P1 animals from entity descriptions only:", or_line(Ae, "neg ~ nox + log_words", "nox"))
D = D.assign(type_c=D["location_type"].where(D["location_type"].map(top) >= 40, "rare_type"))
print("P3 + location type:            ", or_line(D, "neg ~ dec + log_words + C(type_c)", "dec"))

P1 + location type:             OR 3.88 [0.97-15.53], p(two-sided) = 0.0554, n = 129
P1 excluding locations with a 'threat' entity: OR 4.20 [1.22-14.40], p(two-sided) = 0.0226, n = 122
P1 animals from entity descriptions only: OR 8.82 [1.02-76.08], p(two-sided) = 0.0478, n = 66
P3 + location type:             OR 0.27 [0.08-0.87], p(two-sided) = 0.0281, n = 288


/usr/local/lib/python3.11/dist-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(
/usr/local/lib/python3.11/dist-packages/statsmodels/discrete/discrete_model.py:268: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


## Findings

| # | Prediction | Result | Verdict |
|---|---|---|---|
| **P1** | Noxious animals → more negative atmosphere than gentle animals | 84.8% vs 58.3% negative (33 vs 96 locations; 105 dreams); OR 4.08 [1.20–13.90], Holm p = 0.036 | **Hit** |
| **P2** | Noxious animals → more often underground | 25.0% vs 23.6% underground (20 vs 55 located); OR 1.04 [0.31–3.54], p = 0.47 | **Miss** |
| **P3** | Deceased present → less negative atmosphere than living known persons | 45.5% vs 68.8% negative (22 vs 266 locations; deceased in 18 dreams); OR 0.36 [0.15–0.87], Holm p = 0.036 | **Hit** |
| **P4** | Authors meet consistent atmospheres across dreams | ICC1 = 0.094 (permutation null 95th percentile 0.072), Holm p = 0.036; 563 dreams, 181 authors | **Hit** |

**What changed relative to the archived reports.**
- **Animals.** The archived thesis reported that animals are "completely independent of environmental atmosphere (χ² ≈ 0, p = 1.0)". It reinterpreted that as animals expressing the dreamer rather than the environment. That result came from the join bug (notebook 01, D6). Estimated correctly and pre-registered, the animal class tracks the atmosphere of the scene, as Swedenborg's rule predicts: harmful animals where the scene is negative, gentle ones where it is less so. Two parts of the archived claim do not survive:
  - "independence from atmosphere" is withdrawn;
  - the reinterpretation built on it is unnecessary.
- **Animals and height.** Animal class is not related to height (P2 miss). This matches the absence of an elevation gradient in notebook 02.
- **The deceased.** The deceased are associated with less negative scenes than living family and friends. The sample is small (18 dreams), and the interval is wide.
- **Atmosphere follows the person.** Authors meet somewhat consistent atmospheres across different dreams (ICC 0.09). This is modest, and far below the archived "dreamer identity explains 56.4% of variance", which confused dream with dreamer (notebook 04).

**Sensitivity.**
- P1 keeps its direction and size when locations with a `threat` entity are excluded (OR 4.20) and when animals are taken from entity descriptions only (OR 8.82, n = 66). With location type added, OR = 3.88, two-sided p = 0.055.
- P3 holds with location type added (OR 0.27).

**Interpretation (pre-registered limits).** These hits support **pattern fit**: the dreams behave as the framework's correspondences predict on three registered tests out of three primary ones. They do not discriminate the framework from ordinary associations. Vermin are unpleasant, deceased loved ones are emotionally significant, and writing style is stable. All four tests rest on GPT-5.2 codes of the same text (reliability in notebook 06). Samples are small, so the effect sizes are imprecise.